In [37]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [38]:
stocks = [
    "ADANIENT.NS",
    "ADANIPORTS.NS",
    "APOLLOHOSP.NS",
    "ASIANPAINT.NS",
    "AXISBANK.NS",
    "BAJAJ-AUTO.NS",
    "BAJFINANCE.NS",
    "BEL.NS",
    "BHARTIARTL.NS",
    "CIPLA.NS",
    "COALINDIA.NS",
    "DRREDDY.NS",
    "EICHERMOT.NS",
    "ETERNAL.NS",   # formerly Zomato
    "GRASIM.NS",
    "HCLTECH.NS",
    "HDFCBANK.NS",
    "HDFCLIFE.NS",
    "HEROMOTOCO.NS",
    "HINDALCO.NS",
    "HINDUNILVR.NS",
    "ICICIBANK.NS",
    "INDUSINDBK.NS",
    "INFY.NS",
    "ITC.NS",
    "JIOFIN.NS",
    "JSWSTEEL.NS",
    "KOTAKBANK.NS",
    "LT.NS",
    "M&M.NS",
    "MARUTI.NS",
    "NESTLEIND.NS",
    "NTPC.NS",
    "ONGC.NS",
    "POWERGRID.NS",
    "RELIANCE.NS",
    "SBILIFE.NS",
    "SBIN.NS",
    "SHRIRAMFIN.NS",
    "SUNPHARMA.NS",
    "TATACONSUM.NS",
    "TATAMOTORS.NS",
    "TATASTEEL.NS",
    "TCS.NS",
    "TECHM.NS",
    "TITAN.NS",
    "TRENT.NS",
    "ULTRACEMCO.NS",
    "WIPRO.NS"
]




In [39]:
data = yf.download(
    stocks,
    start="2020-01-01",
   
    auto_adjust=True,
    progress=False
)["Close"]

data.head()

$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


Ticker,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJFINANCE.NS,BEL.NS,BHARTIARTL.NS,CIPLA.NS,...,SUNPHARMA.NS,TATACONSUM.NS,TATAMOTORS.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,TRENT.NS,ULTRACEMCO.NS,WIPRO.NS
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,205.794510,361.062622,1403.106812,1690.746582,745.586914,2599.727295,413.532166,29.713718,433.316437,449.420258,...,406.073975,306.714386,NaN,38.457085,1841.149536,602.617432,1132.704834,526.402710,3942.765137,113.709190
2020-01-02,209.111359,366.321014,1470.294189,1688.342285,753.802612,2575.711670,414.973694,30.603798,435.132721,447.153809,...,406.681793,306.240784,NaN,39.862995,1832.698120,605.740845,1133.538452,538.030884,4117.158203,113.984612
2020-01-03,206.240051,365.699554,1461.883301,1651.334717,739.860840,2535.314209,409.832977,29.951080,435.037109,443.801392,...,415.704620,301.363312,NaN,39.768448,1869.222168,612.897034,1117.942017,532.614441,4092.328857,115.269981
2020-01-06,197.576584,363.500580,1438.667969,1609.612915,720.242737,2506.924316,390.604309,28.764307,429.827393,440.779419,...,411.356812,295.586151,NaN,38.909279,1869.052246,609.180603,1136.481201,526.700928,4032.095947,115.751999
2020-01-07,202.032074,367.898590,1454.603882,1625.877319,722.732300,2507.172119,391.674469,28.586288,425.477936,442.526428,...,417.387634,298.237915,NaN,39.143597,1873.639160,614.478577,1137.805420,529.682434,4114.539551,117.152145


In [40]:
returns = data.pct_change().fillna(0)

momentum = data.pct_change(63).shift(1)

volatility = returns.rolling(63).std().shift(1)

rebalance_dates = momentum.index[63::63]

In [41]:
portfolio_returns = []
holdings_history = []

for i in range(len(rebalance_dates) - 1):

    current_date = rebalance_dates[i]
    next_date = rebalance_dates[i + 1]

    # Momentum ranking
    mom = momentum.loc[current_date].dropna()

    # Top 10 momentum stocks
    top10_mom = mom.nlargest(10)

    # Volatility filter
    vol = volatility.loc[current_date]

    candidate_vol = vol[top10_mom.index].dropna()

    # Select 5 lowest volatility stocks
    top5 = candidate_vol.nsmallest(5).index

    holdings_history.append({
        "Date": current_date,
        "Portfolio": list(top5)
    })

    # Portfolio returns
    period_returns = (
        returns.loc[current_date:next_date, top5]
        .mean(axis=1)
    )

    portfolio_returns.append(period_returns)

In [42]:
portfolio_returns = pd.concat(portfolio_returns)

portfolio_returns = portfolio_returns[
    ~portfolio_returns.index.duplicated()
]

equity_curve = (1 + portfolio_returns).cumprod()

benchmark_returns = returns.mean(axis=1)

benchmark_curve = (1 + benchmark_returns).cumprod()

In [43]:
years = (
    portfolio_returns.index[-1]
    - portfolio_returns.index[0]
).days / 365.25

cagr = (
    equity_curve.iloc[-1]
    ** (1 / years)
    - 1
)

sharpe = (
    portfolio_returns.mean()
    / portfolio_returns.std()
) * np.sqrt(252)

drawdown = (
    equity_curve
    / equity_curve.cummax()
    - 1
)

max_drawdown = drawdown.min()


print(f"Strategy Return: {(equity_curve.iloc[-1]-1)*100:.2f}%")
print(f"Benchmark Return: {(benchmark_curve.iloc[-1]-1)*100:.2f}%")
print(f"CAGR: {cagr*100:.2f}%")
print(f"Sharpe: {sharpe:.2f}")
print(f"Max Drawdown: {max_drawdown*100:.2f}%")

Strategy Return: 214.89%
Benchmark Return: 241.03%
CAGR: 20.64%
Sharpe: 1.23
Max Drawdown: -17.26%
